In [ ]:
# 1. FORCE JUPYTER TO RELOAD CHANGED .py FILES
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.tca_math import HierarchicalTCAFuser

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Point to the master Parquet file with in-situ data
INPUT_PARQUET = PROJECT_ROOT / "outputs" / "processed" / "sarawak_MASTER_with_insitu.parquet"

# Output file for the TCA-fused dataset
OUTPUT_PARQUET = PROJECT_ROOT / "outputs" / "processed" / "sarawak_TCA_fused.parquet"

# ==========================================
# 2. INITIALIZE & RUN TCA
# ==========================================
fuser = HierarchicalTCAFuser(str(INPUT_PARQUET))

# Execute the fusion
df_tca = fuser.run_fusion(str(OUTPUT_PARQUET))

# ==========================================
# 3. INSPECT THE RESULTS
# ==========================================
print("\n--- First 5 Rows of TCA-Fused Data ---")
cols_to_show = ['station_id', 'date', 'precip_chirps', 'precip_era5_land', 
                'precip_persiann', 'P_merged_global', 'P_merged_hier']
print(df_tca[cols_to_show].head())

print("\n--- Comparison of P_merged Methods (Mean Monthly Precipitation) ---")
print(df_tca[['P_merged_global', 'P_merged_season', 'P_merged_elev', 'P_merged_hier']].mean())

print("\n--- Missing Values in P_merged ---")
print(df_tca[['P_merged_global', 'P_merged_season', 'P_merged_elev', 'P_merged_hier']].isna().sum())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🔄 Loading master dataset for TCA...
✅ Data loaded: 170678 rows, 335 stations.
   Seasons: <ArrowStringArray>
['NEM', 'Inter-Monsoon', 'SWM']
Length: 3, dtype: str
   Elevation Bins: <ArrowStringArray>
['Q4', 'Q3', 'Q2', 'Q1']
Length: 4, dtype: str

 Starting Hierarchical TCA Fusion...
   -> Calculating Global weights...
   -> Calculating Seasonal weights...
   -> Calculating Elevation weights...
   -> Calculating Hierarchical weights (Season x Elevation)...
   -> Mapping weights and calculating P_merged products...

✅ SUCCESS! Saved TCA-fused dataset to: d:\my_project\outputs\processed\sarawak_TCA_fused.parquet
   Shape: 170678 rows × 20 columns

📊 TCA Weight Summary (Multiplicative Model):
   Global: CHIRPS=0.351, PERSIANN=0.445, ERA5=0.205
   NEM: CHIRPS=0.378, PERSIANN=0.414, ERA5=0.208
   Inter-Monsoon: CHIRPS=0.434, PERSIANN=0.382, ERA5=0.185
   SWM: CHIRPS=0.310, PERSIANN=0.515, ERA5=0.175

--

Using `polars` to convert parquet into CSV

In [3]:
import polars as pl

# Read and write efficiently
df = pl.read_parquet(r'D:\my_project\outputs\processed\sarawak_TCA_fused.parquet')
df.write_csv(r'D:\my_project\outputs\processed\sarawak_TCA_fused.csv')